# H&M Socks — Exploratory Data Analysis

**Project:** RAG-based retrieval system for H&M socks products.

This notebook covers the exploratory data analysis (EDA) stage of the project. We load the raw H&M "Personalized Fashion Recommendations" dataset, inspect its structure and quality, and narrow the scope down to the **Socks & Tights** product family that the downstream RAG pipeline (see `02_RAG_HM_Socks.ipynb`) is built on.

The goal here is purely to understand the data — no retrieval, embedding, or indexing logic lives in this notebook.

## 1. Import Libraries & Load Datasets

We load the three core H&M tables: `articles` (product catalog), `customers`, and `transactions`. For this project only `articles` is used going forward, but all three are loaded here as part of the original data-loading step.

In [2]:
import pandas as pd
import numpy as np

BASE_PATH = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations"

articles = pd.read_csv(f"{BASE_PATH}/articles.csv")
customers = pd.read_csv(f"{BASE_PATH}/customers.csv")
transactions = pd.read_csv(f"{BASE_PATH}/transactions_train.csv")

## 2. Dataset Overview

A quick look at the shape of each table and a preview of their columns/rows.

In [3]:
print("Articles:", articles.shape)
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)

Articles: (105542, 25)
Customers: (1371980, 7)
Transactions: (31788324, 5)


In [4]:
articles.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [5]:
customers.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24.0,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54.0,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52.0,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [6]:
transactions.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## 3. Shapes and Columns

Full column listing for each table, to understand what fields are available.

In [10]:
print("ARTICLE COLUMNS")
print(articles.columns.tolist())

print("\nCUSTOMER COLUMNS")
print(customers.columns.tolist())

print("\nTRANSACTION COLUMNS")
print(transactions.columns.tolist())

ARTICLE COLUMNS
['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc']

CUSTOMER COLUMNS
['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']

TRANSACTION COLUMNS
['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']


## 4. Data Types

`articles.info()` shows dtypes and non-null counts, which feeds directly into the missing-value analysis below.

In [8]:
articles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105542 entries, 0 to 105541
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   article_id                    105542 non-null  int64 
 1   product_code                  105542 non-null  int64 
 2   prod_name                     105542 non-null  object
 3   product_type_no               105542 non-null  int64 
 4   product_type_name             105542 non-null  object
 5   product_group_name            105542 non-null  object
 6   graphical_appearance_no       105542 non-null  int64 
 7   graphical_appearance_name     105542 non-null  object
 8   colour_group_code             105542 non-null  int64 
 9   colour_group_name             105542 non-null  object
 10  perceived_colour_value_id     105542 non-null  int64 
 11  perceived_colour_value_name   105542 non-null  object
 12  perceived_colour_master_id    105542 non-null  int64 
 13 

## 5. Missing Values

Column-by-column count of missing values in `articles`.

In [9]:
articles.isna().sum().sort_values(ascending=False)

detail_desc                     416
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
article_id                        0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_name       0
perceived_colour_value_id         0
perceived_colour_master_name      0
department_no                     0
department_name                   0
perceived_colour_master_id        0
index_code                        0
index_name                        0
index_group_name                  0
index_group_no                    0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
dtype: int64

## 6. Unique Values / Cardinality & Article Statistics

Cardinality of `article_id` vs `product_code` (this distinction matters a lot for the RAG design — see the note below), plus a breakdown of product groups and the most common product types.

In [11]:
# Basic article statistics

print("Unique article IDs:", articles["article_id"].nunique())
print("Unique product codes:", articles["product_code"].nunique())

print("\nMissing values:")
print(
    articles.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nProduct groups:")
print(
    articles["product_group_name"]
    .value_counts()
)

print("\nTop product types:")
print(
    articles["product_type_name"]
    .value_counts()
    .head(30)
)

Unique article IDs: 105542
Unique product codes: 47224

Missing values:
detail_desc                     416
product_code                      0
prod_name                         0
product_type_no                   0
product_type_name                 0
product_group_name                0
graphical_appearance_no           0
graphical_appearance_name         0
article_id                        0
colour_group_code                 0
colour_group_name                 0
perceived_colour_value_name       0
perceived_colour_value_id         0
perceived_colour_master_name      0
department_no                     0
department_name                   0
perceived_colour_master_id        0
index_code                        0
index_name                        0
index_group_name                  0
index_group_no                    0
section_no                        0
section_name                      0
garment_group_no                  0
garment_group_name                0
dtype: int64

Product groups

> **Why this matters for the RAG project:** `article_id` identifies a single color/size *variant*, while `product_code` identifies the *product/style* that groups those variants together. The dataset has far more unique `article_id` values than unique `product_code` values — this is exactly why the RAG pipeline retrieves at the `product_code` level and resolves down to `article_id` only at the very end (see Notebook 2, Section 3).

## 7. Socks Filtering

We narrow the catalog down to sock-related products by filtering `product_type_name` for the substring "sock".

In [12]:
socks = articles[
    articles["product_type_name"]
    .str.contains("sock", case=False, na=False)
]

print("Sock articles:", len(socks))
print("Unique sock products:", socks["product_code"].nunique())

display(
    socks[
        [
            "article_id",
            "product_code",
            "prod_name",
            "product_type_name",
            "product_group_name",
            "colour_group_name",
            "department_name",
            "section_name",
            "garment_group_name",
            "detail_desc"
        ]
    ].head(20)
)

Sock articles: 1889
Unique sock products: 474


,article_id,product_code,prod_name,product_type_name,product_group_name,colour_group_name,department_name,section_name,garment_group_name,detail_desc
7,111565003,111565,20 den 1p Stockings,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Semi shiny nylon stockings with a wide, reinfo..."
54,148033006,148033,Nouvelle 1p Stay Up,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi shiny stay-ups with a wide lace trim at t...
63,156224001,156224,Box 4p Socks,Socks,Socks & Tights,Beige,Tights basic,"Womens Nightwear, Socks & Tigh",Socks and Tights,Semi-matte socks with a short shaft. 20 denier.
74,160442007,160442,3p Sneaker Socks,Socks,Socks & Tights,Black,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
75,160442010,160442,3p Sneaker Socks,Socks,Socks & Tights,White,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
76,160442042,160442,Sneaker 3p Socks,Socks,Socks & Tights,Light Grey,Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
77,160442043,160442,3p Sneaker Socks,Socks,Socks & Tights,Light Grey,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Socks and Tights,"Short, fine-knit socks designed to be hidden b..."
78,162074062,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Pink,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
79,162074069,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Grey,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.
80,162074071,162074,Sigge sneaker sock 5p,Socks,Socks & Tights,Black,UW,Divided Basics,"Under-, Nightwear",Fine-knit trainer socks in various colours.


**Finding:** filtering on `product_type_name` gives us the working sock dataset used throughout the rest of the project.

## 8. Socks Dataset Analysis (Product / Variant Analysis)

Grouping the sock articles by `product_code` shows how many color/size variants each sock product has.

In [13]:
sock_product_stats = (
    socks.groupby("product_code")
    .agg(
        variant_count=("article_id", "nunique"),
        product_name=("prod_name", "first"),
        product_type=("product_type_name", "first"),
        description=("detail_desc", "first")
    )
    .sort_values("variant_count", ascending=False)
)

display(sock_product_stats.head(20))

,variant_count,product_name,product_type,description
product_code,,,,
783707,75,1pk Fun,Socks,"Socks in a soft, jacquard-knit cotton blend wi..."
807241,38,1p Socks,Socks,"Socks in a soft, jacquard-knit cotton blend wi..."
679687,31,Eivor 7-p shaftless SG,Socks,Fine-knit trainer socks in a soft cotton blend...
679709,31,Eivor 7-p shaftless,Socks,Fine-knit trainer socks in a soft cotton blend...
619764,30,Marie 7-p. ankle sock,Socks,Fine-knit socks in a soft cotton blend.
584631,30,5pk regular fancy,Socks,"Socks in a soft, fine-knit cotton blend with e..."
423900,27,Gullan 5-p ankle sock,Socks,Jacquard-knit socks in various colours and des...
683001,26,Pattern 7p Socks,Socks,Fine-knit socks in a soft cotton blend.
631536,25,Basic 7p shaftless pattern,Socks,Fine-knit trainer socks in a soft cotton blend...


In [14]:
print("Products with 1 variant:",
      (sock_product_stats["variant_count"] == 1).sum())

print("Products with >1 variant:",
      (sock_product_stats["variant_count"] > 1).sum())

print("Max variants for one product:",
      sock_product_stats["variant_count"].max())

Products with 1 variant: 196
Products with >1 variant: 278
Max variants for one product: 75


**Finding:** most sock products have multiple variants, and a handful of products (e.g. `1pk Fun`) have a very large number of variants. This confirms that product-level retrieval (over `product_code`) is the right granularity for semantic search, with variant-level (`article_id`) lookup handled separately.

## 9. Gender Analysis

`articles` does not have a direct "gender" field for socks, so gender is **inferred** from `index_name`, `index_group_name`, `department_name`, and `section_name`.

In [33]:
import re

def infer_gender(row):
    text = " ".join([
        str(row["index_name"]),
        str(row["index_group_name"]),
        str(row["department_name"]),
        str(row["section_name"])
    ]).lower()

    # Women
    if (
        "ladieswear" in text
        or "womens" in text
        or "women" in text
        or "ladies" in text
    ):
        return "Women"

    # Men
    if (
        "menswear" in text
        or "mens" in text
        or re.search(r"\bmen\b", text)
    ):
        return "Men"

    return "Unisex/Unknown"


socks["inferred_gender"] = socks.apply(
    infer_gender,
    axis=1
)

print(socks["inferred_gender"].value_counts())

inferred_gender
Unisex/Unknown    1008
Women              468
Men                413
Name: count, dtype: int64


/tmp/ipykernel_58/1363076787.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  socks["inferred_gender"] = socks.apply(


In [34]:
display(
    socks[
        [
            "article_id",
            "product_code",
            "prod_name",
            "index_name",
            "index_group_name",
            "department_name",
            "section_name",
            "inferred_gender"
        ]
    ].sample(30, random_state=42)
)

,article_id,product_code,prod_name,index_name,index_group_name,department_name,section_name,inferred_gender
57086,715652004,715652,3p Socks,Lingeries/Tights,Ladieswear,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Women
45368,683001029,683001,Pattern 7p Socks,Lingeries/Tights,Ladieswear,Shopbasket Socks,"Womens Nightwear, Socks & Tigh",Women
93669,859174006,859174,Bosse 5-P,Baby Sizes 50-98,Baby/Children,Baby Socks,Baby Essentials & Complements,Unisex/Unknown
93674,859174013,859174,Bosse 5-P,Baby Sizes 50-98,Baby/Children,Baby Socks,Baby Essentials & Complements,Unisex/Unknown
73966,774201001,774201,X-mas sock 3-pack,Children Sizes 92-140,Baby/Children,Kids Boy Socks,Boys Underwear & Basics,Unisex/Unknown
92975,855989001,855989,Shaftless solid 15PACK,Baby Sizes 50-98,Baby/Children,Baby Socks,Baby Essentials & Complements,Unisex/Unknown
19576,584631018,584631,5pk regular fancy,Menswear,Menswear,Socks Bin,Men Underwear,Men
96689,871241001,871241,Basic rolltops 5PACK,Baby Sizes 50-98,Baby/Children,Baby Socks,Baby Essentials & Complements,Unisex/Unknown
37263,653188018,653188,Sigge sneaker sock 5p.,Divided,Divided,UW,Divided Basics,Unisex/Unknown
3291,452851017,452851,Marie 7-p sock,Children Sizes 134-170,Baby/Children,Young Girl S&T,Girls Underwear & Basics,Unisex/Unknown


In [35]:
print(
    pd.crosstab(
        socks["index_group_name"],
        socks["inferred_gender"]
    )
)

inferred_gender   Men  Unisex/Unknown  Women
index_group_name                            
Baby/Children       0             894      0
Divided             0              88      0
Ladieswear          0               0    402
Menswear          362               0      0
Sport              51              26     66


**Finding:** the inferred gender is unevenly distributed across the sock catalog, with a large "Unisex/Unknown" bucket. This inferred `inferred_gender` field is later carried into the RAG product documents as a filterable metadata field (see Notebook 2, Section 4).

## 10. Relevant Data Quality Findings

As an alternative to filtering by `product_type_name`, we also check filtering directly on `product_group_name == "Socks & Tights"` after cleaning up text fields (`detail_desc`, `prod_name`). This is a data-quality check on the filtering approach, not a redo of the main sock dataset used later.

In [15]:
articles_clean = articles.copy()

articles_clean["detail_desc"] = (
    articles_clean["detail_desc"]
    .fillna("")
    .str.strip()
)

articles_clean["prod_name"] = (
    articles_clean["prod_name"]
    .fillna("")
    .str.strip()
)

In [16]:
sock_articles = articles_clean[
    articles_clean["product_group_name"].eq("Socks & Tights")
].copy()

print(sock_articles.shape)
print(sock_articles["product_type_name"].value_counts())

(2442, 25)
product_type_name
Socks               1889
Underwear Tights     546
Leg warmers            7
Name: count, dtype: int64


## 11. EDA Conclusions

- The H&M catalog contains **1,889 sock article variants** spanning **474 unique sock products** (`product_code`), confirming a clear variant-vs-product distinction that shapes the RAG retrieval design.
- Sock products vary widely in how many color/size variants they have, with some products (e.g. `1pk Fun`, product_code `783707`) having dozens of variants.
- There is no explicit gender field for socks in the raw data; gender must be **inferred** from index/department/section text, and a large portion of products end up "Unisex/Unknown".
- An alternative filter on `product_group_name == "Socks & Tights"` was checked as a data-quality cross-check against the `product_type_name`-based filter used for the main sock dataset.

These findings directly inform the RAG pipeline in `02_RAG_HM_Socks.ipynb`: product-level documents are built per `product_code`, gender/color/type are kept as structured metadata for hybrid filtering, and `article_id` is resolved only at the final variant-lookup step.